### SCD Type 1
- **Purpose**: Updates records with new data, without keeping history.
- **Behavior**: Old data is simply replaced with new data.
- **Key Point**: No history is kept of the previous data; only the current data is available.

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS clever_db;


In [0]:
%sql
CREATE TABLE clever_db.delta_load_type1_tbl (
    EmployeeID INT,
    Name STRING,
    Salary INT,
    Designation STRING,
    City STRING
)
USING DELTA;


In [0]:
%sql
INSERT INTO clever_db.delta_load_type1_tbl VALUES
(100, 'David', 85000, 'Developer', 'Chicago'),
(101, 'Tim', 75000, 'Lead', 'Texas');


num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
select * from clever_db.delta_load_type1_tbl;

EmployeeID,Name,Salary,Designation,City
100,David,85000,Developer,Chicago
101,Tim,75000,Lead,Texas


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define the schema for the DataFrame
schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Salary", IntegerType(), True),
    StructField("Designation", StringType(), True),
    StructField("City", StringType(), True)
])

# Create the data
data = [
    (102, 'Charlie', 75000, 'QA', 'Texas'),
    (101, 'Tim', 80000, 'Lead', 'Texas'),
    (103, 'Sophia', 92000, 'Analyst', 'New York')
]

# Create a DataFrame using the data and schema
df = spark.createDataFrame(data, schema)

# Show the DataFrame
df.show()


+----------+-------+------+-----------+--------+
|EmployeeID|   Name|Salary|Designation|    City|
+----------+-------+------+-----------+--------+
|       102|Charlie| 75000|         QA|   Texas|
|       101|    Tim| 80000|       Lead|   Texas|
|       103| Sophia| 92000|    Analyst|New York|
+----------+-------+------+-----------+--------+



###Type-1 Implementation

In [0]:
df.createOrReplaceTempView("source_view")

In [0]:
%sql
select * from source_view;

EmployeeID,Name,Salary,Designation,City
102,Charlie,75000,QA,Texas
101,Tim,80000,Lead,Texas
103,Sophia,92000,Analyst,New York


In [0]:
%sql
select * from clever_db.delta_load_type1_tbl;

In [0]:
%sql
MERGE INTO clever_db.delta_load_type1_tbl AS target
USING source_view AS source
ON target.EmployeeID = source.EmployeeID
WHEN MATCHED 
THEN UPDATE SET
    target.Name = source.Name,
    target.Salary = source.Salary,
    target.Designation = source.Designation,
    target.City = source.City
WHEN NOT MATCHED 
THEN INSERT (EmployeeID, Name, Salary, Designation, City) 
VALUES (source.EmployeeID, source.Name, source.Salary, source.Designation, source.City);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,1,0,2


In [0]:
%sql
select * from clever_db.delta_load_type1_tbl;

EmployeeID,Name,Salary,Designation,City
103,Sophia,92000,Analyst,New York
100,David,85000,Developer,Chicago
102,Charlie,75000,QA,Texas
101,Tim,80000,Lead,Texas


In [0]:
'SCD1 no history maintained, records are updated and new records are inserted'

'SCD1 no history maintained, records are updated and new records are inserted'

In [0]:
No history is kept of the previous data; only the current data is available.

In [0]:
spark.sql("DELETE FROM clever_db.delta_load_type1_tbl WHERE EmployeeID >= 102")

In [0]:
%sql
--DROP TABLE IF EXISTS clever_db.delta_load_type1_tbl

In [0]:
#dbutils.fs.rm("dbfs:/input/clever_db/delta_load_type1_tbl/", recurse=True)